# Data modeling 10: interview case studies

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/data-modeling/data-modeling-10-interview-case-studies.ipynb)

Data modeling interviews give a business and a few questions, and ask you to design the tables. Interviewers
are not looking for one right schema. They watch whether you ask about the questions first, declare a grain,
separate facts from dimensions, handle history and many-to-many relationships deliberately, and can show that
the design answers the questions.

This lesson gives a method, then works through four classic cases: ride sharing, a subscription business,
returns for the shop, and music streaming. Each case builds a small version of the model and runs the query it
was designed for. It ends with short answers to questions that come up in almost every modeling interview.

**Prerequisites:** Lessons 1 to 4 ([Data modeling 1: grain, facts and dimensions](data-modeling-01-grain-facts-and-dimensions.ipynb), [Data modeling 2: star schema design](data-modeling-02-star-schema-design.ipynb), [Data modeling 3: slowly changing dimensions](data-modeling-03-slowly-changing-dimensions.ipynb), [Data modeling 4: relationships and double counting](data-modeling-04-relationships-and-double-counting.ipynb)). Lessons 7 and 9 help with follow-up questions.

*Last reviewed: November 2026, against DuckDB 2.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q duckdb~=2.0

## Setup and sample data

The next cell creates an in-memory DuckDB database and loads `src_customers` (the CRM's current view of each customer), `src_products` (the product catalogue), `src_orders` (one row per order, with fulfilment timestamps) and `src_order_lines` (one row per product on an order): extracts from a small online shop that sells coffee, tea and kitchenware. Every lesson starts from these source tables; [Data modeling 0](data-modeling-00-start-here.ipynb) describes them.

In [ ]:
import duckdb

con = duckdb.connect()  # an in-memory database: the warehouse for this notebook
con.sql("""
create table src_customers (
    customer_id int primary key, name varchar, email varchar, city varchar, segment varchar, created_on date
);
insert into src_customers values
    (101, 'Acme Corp',     'ops@acme.example',     'Leeds', 'smb',        '2025-11-02'),
    (102, 'Bright Retail', 'buy@bright.example',   'York',  'enterprise', '2025-11-20'),  -- was smb until Feb 1
    (103, 'Cyan Traders',  'hi@cyan.example',      'Bath',  'enterprise', '2025-12-05'),
    (104, 'Delta Grocers', 'orders@delta.example', 'Leeds', 'smb',        '2026-01-08'),  -- moved from Hull on Feb 15
    (105, 'Echo Supplies', null,                   'Derby', 'smb',        '2026-01-19'),
    (106, 'Foxtrot Foods', 'team@foxtrot.example', 'York',  'mid-market', '2026-02-10');  -- has not ordered yet
create table src_products (
    sku varchar primary key, product_name varchar, category varchar, department varchar, list_price decimal(8, 2)
);
insert into src_products values
    ('SKU-01', 'House Blend 1kg',    'Coffee',  'Grocery', 18.00),
    ('SKU-02', 'Espresso Beans 1kg', 'Coffee',  'Grocery', 22.00),
    ('SKU-03', 'Green Tea 100 bags', 'Tea',     'Grocery',  6.50),
    ('SKU-04', 'Earl Grey 100 bags', 'Tea',     'Grocery',  5.50),
    ('SKU-05', 'Stoneware Mug',      'Mugs',    'Kitchen',  9.00),
    ('SKU-06', 'Travel Mug',         'Mugs',    'Kitchen', 15.00),
    ('SKU-07', 'Gooseneck Kettle',   'Kettles', 'Kitchen', 65.00),
    ('SKU-08', 'Electric Kettle',    'Kettles', 'Kitchen', 45.00);
create table src_orders (
    order_id int primary key, customer_id int, order_ts timestamp, channel varchar, status varchar,
    shipping_fee decimal(8, 2), shipped_ts timestamp, delivered_ts timestamp
);
insert into src_orders values
    (1001, 101, '2026-01-05 09:12', 'web',   'delivered', 4.99, '2026-01-06 10:00', '2026-01-08 15:00'),
    (1002, 102, '2026-01-09 14:30', 'store', 'delivered', 0.00, '2026-01-09 14:30', '2026-01-09 14:30'),
    (1003, 103, '2026-01-14 11:05', 'web',   'delivered', 0.00, '2026-01-15 09:00', '2026-01-19 12:00'),
    (1004, 101, '2026-01-28 16:40', 'web',   'delivered', 4.99, '2026-01-29 11:00', '2026-02-02 10:30'),
    (1005, 104, '2026-02-03 10:20', 'web',   'delivered', 4.99, '2026-02-04 08:45', '2026-02-06 16:10'),
    (1006, 102, '2026-02-11 13:15', 'web',   'shipped',   0.00, '2026-02-12 09:30', null),
    (1007, 105, '2026-02-14 09:50', 'store', 'delivered', 0.00, '2026-02-14 09:50', '2026-02-14 09:50'),
    (1008, 104, '2026-02-20 18:05', 'web',   'placed',    4.99, null,               null),
    (1009, 103, '2026-02-24 12:00', 'web',   'cancelled', 4.99, null,               null),
    (1010, 101, '2026-02-27 08:30', 'store', 'delivered', 0.00, '2026-02-27 08:30', '2026-02-27 08:30');
create table src_order_lines (
    order_id int, line_no int, sku varchar, qty int, unit_price decimal(8, 2), discount decimal(8, 2),
    primary key (order_id, line_no)
);
insert into src_order_lines values
    (1001, 1, 'SKU-01',  2, 18.00,  0.00), (1001, 2, 'SKU-05', 4,  9.00, 0.00),
    (1002, 1, 'SKU-07',  1, 65.00,  5.00),
    (1003, 1, 'SKU-02', 10, 22.00, 20.00), (1003, 2, 'SKU-03', 5,  6.50, 0.00),
    (1004, 1, 'SKU-06',  2, 15.00,  0.00),
    (1005, 1, 'SKU-01',  3, 18.00,  0.00), (1005, 2, 'SKU-04', 2,  5.50, 0.00), (1005, 3, 'SKU-08', 1, 45.00, 0.00),
    (1006, 1, 'SKU-02', 20, 22.00, 40.00),
    (1007, 1, 'SKU-05',  6,  9.00,  0.00),
    (1008, 1, 'SKU-03',  4,  6.50,  0.00),
    (1009, 1, 'SKU-07',  1, 65.00,  0.00),
    (1010, 1, 'SKU-01',  1, 18.00,  0.00), (1010, 2, 'SKU-06', 1, 15.00, 0.00);
""")
con.sql("show tables").show()

## A method for any modeling question

Work through these steps out loud. Writing the grain sentence and the questions on the whiteboard before any table
is the single most useful habit.

1. **Clarify the questions.** Ask for three to five concrete questions the model must answer, who asks them, and how
   fresh the answers must be. Every later choice is justified by these.
2. **Name the business processes.** Each process that produces events becomes a fact table: trips, payments,
   subscription changes.
3. **Declare the grain** of each fact in one sentence, at the finest level the source records.
4. **List dimensions** (who, what, where, when, how) and decide **history** per attribute: Type 1 or Type 2.
5. **List measures** and their additivity. Put coarser-grain measures on a coarser fact.
6. **Find the traps**: many-to-many relationships (bridge), several dates (role-playing), levels such as balances
   (periodic snapshot), processes with milestones (accumulating snapshot), late or early data, deletes.
7. **Prove it**: write the SQL for one or two of the questions against your tables.
8. **Scale and operate**: partitioning, clustering, incremental loads, tests.

Interviewers often change a requirement halfway ("now finance wants it by the region at the time"). That is a
history question; answer it by changing the attribute to Type 2 and the join to a point-in-time join, not by starting
over.

## Case 1: ride sharing

**Prompt.** Design a model for a ride-sharing company. Questions: trips and revenue per city per week; driver
earnings per week; cancellation rate by city; average wait time from request to pickup.

**Questions to ask.** Is revenue the rider's fare or the company's take? Do drivers change city or vehicle?
Can a trip be cancelled after pickup? Are tips part of earnings?

**Design.**

- Process: trip requests. Grain: **one row per trip request**, including cancelled ones (cancellation rate
  needs them).
- `fact_trip`: request, accept, pickup and dropoff timestamps (an accumulating snapshot: the row is updated as
  the trip progresses), status, fare, tip, driver payout, distance, and keys.
- Dimensions: `dim_rider`, `dim_driver` (Type 2 on home city and vehicle class), `dim_location` used in two
  roles (pickup and dropoff), `dim_date`.
- Measures: fare, tip and payout are additive; wait time is stored per trip, so averages are recomputed from
  trips, never averaged from averages; cancellation rate is cancelled trips over requested trips.

**Where it matters**
- Any marketplace: delivery, freelancing, rentals.

**Watch out for**
- Dropping cancelled requests from the fact: the cancellation-rate question becomes impossible.
- Attributing trips to the driver's current city instead of the trip's pickup city.

### Example

A small `fact_trip` and the city-level questions.

In [ ]:
con.sql("""
create table dim_location (location_key int, zone varchar, city varchar);
insert into dim_location values (1, 'Centre', 'Leeds'), (2, 'Airport', 'Leeds'), (3, 'Old Town', 'York');

create table fact_trip (
    trip_id int, driver_key int, pickup_location_key int, dropoff_location_key int,
    requested_ts timestamp, pickup_ts timestamp, dropoff_ts timestamp, status varchar,
    fare decimal(8, 2), tip decimal(8, 2), driver_payout decimal(8, 2)
);
insert into fact_trip values
    (1, 10, 1, 2, '2026-03-02 08:00', '2026-03-02 08:06', '2026-03-02 08:30', 'completed', 24.00, 3.00, 18.00),
    (2, 11, 1, 1, '2026-03-02 09:00', null,               null,               'cancelled',  0.00, 0.00,  0.00),
    (3, 10, 2, 1, '2026-03-03 18:00', '2026-03-03 18:12', '2026-03-03 18:40', 'completed', 26.50, 0.00, 20.00),
    (4, 12, 3, 3, '2026-03-04 12:00', '2026-03-04 12:04', '2026-03-04 12:15', 'completed',  9.00, 1.00,  7.00),
    (5, 12, 3, 3, '2026-03-04 22:00', null,               null,               'cancelled',  0.00, 0.00,  0.00),
    (6, 12, 3, 3, '2026-03-05 07:30', null,               null,               'cancelled',  0.00, 0.00,  0.00);
""")
con.sql("""
select pu.city,
       count(*)                                                        as requests,
       count(*) filter (where t.status = 'completed')                 as completed,
       round(100.0 * count(*) filter (where t.status = 'cancelled') / count(*), 1) as cancel_rate_pct,
       sum(t.fare)                                                     as fares,
       round(avg(date_diff('minute', t.requested_ts, t.pickup_ts)), 1) as avg_wait_minutes
from fact_trip t join dim_location pu on pu.location_key = t.pickup_location_key   -- location in the pickup role
group by pu.city order by pu.city
""").show()

**What to notice:** Leeds has 3 requests with 1 cancellation (33.3 %), York 3 with 2 (66.7 %). Both rates exist only because cancelled
requests are rows in the fact. The average wait ignores cancelled trips automatically, because their pickup time is
null and `avg` skips nulls: say that rule out loud, since an interviewer may ask whether it is the right one.

## Case 2: subscription business and MRR

**Prompt.** A SaaS company sells monthly plans. Questions: monthly recurring revenue (MRR) by month; MRR
movements (new, expansion, contraction, churn); active customers at month end.

**Design.**

- Two processes, two facts. `fact_subscription_event`: **one row per change** (start, upgrade, downgrade,
  cancel) with the new monthly price, a transaction fact.
- `fact_mrr_monthly`: **one row per customer per month**, the MRR at month end, a periodic snapshot built from
  the events with a spine of months (lesson 7). MRR is semi-additive: sum across customers within a month,
  never across months.
- Movements come from comparing each customer's MRR with the previous month (`lag`): from 0 is new, up is
  expansion, down is contraction, to 0 is churn.
- `dim_plan` is Type 2 if plan prices change, so old events keep their old price.

**Where it matters**
- Subscriptions, memberships, insurance policies, loans: anything whose level matters more than its events.

**Watch out for**
- Computing MRR from invoices: annual plans billed once would look like a spike and eleven empty months.
- Summing monthly MRR over a quarter.

### Example

Build monthly MRR from subscription events and classify each customer's movement.

In [ ]:
con.sql("""
create table fact_subscription_event (customer varchar, event_date date, event varchar, monthly_price decimal(8, 2));
insert into fact_subscription_event values
    ('A', '2026-01-10', 'start',     50), ('A', '2026-03-05', 'upgrade',  80),
    ('B', '2026-01-20', 'start',    100), ('B', '2026-02-15', 'downgrade', 60), ('B', '2026-04-02', 'cancel', 0),
    ('C', '2026-02-01', 'start',     30),
    ('D', '2026-03-12', 'start',     50);

create table fact_mrr_monthly as
with months as (select cast(m as date) as month_end from range(date '2026-01-31', date '2026-05-01', interval 1 month) t(m)),
spine as (select c.customer, m.month_end from (select distinct customer from fact_subscription_event) c cross join months m)
select s.customer, last_day(s.month_end) as month_end,
       coalesce((select arg_max(e.monthly_price, e.event_date) from fact_subscription_event e
                 where e.customer = s.customer and e.event_date <= last_day(s.month_end)), 0) as mrr
from spine s;
""")
con.sql("""
with moves as (
    select month_end, customer, mrr, coalesce(lag(mrr) over (partition by customer order by month_end), 0) as prev
    from fact_mrr_monthly
)
select month_end,
       sum(mrr)                                               as mrr,
       count(*) filter (where mrr > 0)                         as active_customers,
       coalesce(sum(mrr) filter (where prev = 0 and mrr > 0), 0)            as new,
       coalesce(sum(mrr - prev) filter (where prev > 0 and mrr > prev), 0)  as expansion,
       coalesce(sum(mrr - prev) filter (where mrr > 0 and mrr < prev), 0)   as contraction,
       coalesce(-sum(prev) filter (where prev > 0 and mrr = 0), 0)          as churn
from moves group by month_end order by month_end
""").show()

**What to notice:** January's 150 is A (50) and B (100). In February C starts (+30 new) and B downgrades from 100 to 60 (−40
contraction): 150 + 30 − 40 = 140. In March D starts (+50) and A upgrades to 80 (+30 expansion): 220. In April B
cancels (−60 churn): 160. Every month's MRR is the previous month's plus its movements, which is the check finance
will run on your model. The snapshot also makes "active customers at month end" a simple count.

## Case 3: returns for the shop

**Prompt.** The shop now accepts returns. Questions: net revenue (sales minus refunds) by month and category;
return rate by product; which promotions lead to returns.

**Design.**

- A new process, a new fact: `fact_return_line`, **one row per returned order line**, with the refund amount,
  the return date and a reason code. Keep the original `order_id` and `line_no` so each return links to the
  sale.
- Do **not** update or delete the sale. The sale happened; the return is a separate event on a later date.
- Net revenue is a drill-across of two facts (lesson 2): sales by month of sale minus refunds by month of
  return, or, for "how much of January's sales stuck", refunds attributed to the sale's month through the link.
  The business must choose which; offer both.
- Return rate by product: returned units over sold units, each summed separately.

**Where it matters**
- Returns, refunds, chargebacks, credit notes, cancellations after fulfilment.

**Watch out for**
- Subtracting returns from the sales fact in place: history changes, and the return's date and reason are lost.

### Example

Two returns arrive: part of order 1003's espresso line and the whole of order 1005's kettle line. Report net revenue both ways.

In [ ]:
con.sql("""
create table fact_sales as
select l.order_id, l.line_no, l.sku, cast(o.order_ts as date) as sale_date, l.qty, l.qty * l.unit_price - l.discount as net_amount
from src_order_lines l join src_orders o using (order_id) where o.status <> 'cancelled';

create table fact_return_line (order_id int, line_no int, return_date date, qty int, refund decimal(8, 2), reason varchar);
insert into fact_return_line values (1003, 1, '2026-02-02', 3, 60.00, 'damaged'), (1005, 3, '2026-02-09', 1, 45.00, 'changed mind');
""")
con.sql("""
with sales as (select date_trunc('month', sale_date)::date as month, sum(net_amount) as sales from fact_sales group by all),
refunds_by_return_month as (select date_trunc('month', return_date)::date as month, sum(refund) as refunds from fact_return_line group by all),
refunds_by_sale_month as (
    select date_trunc('month', s.sale_date)::date as month, sum(r.refund) as refunds
    from fact_return_line r join fact_sales s using (order_id, line_no) group by all
)
select s.month, s.sales,
       coalesce(rr.refunds, 0) as refunds_dated_by_return,  s.sales - coalesce(rr.refunds, 0) as net_by_return_month,
       coalesce(rs.refunds, 0) as refunds_dated_by_sale,    s.sales - coalesce(rs.refunds, 0) as net_by_sale_month
from sales s
left join refunds_by_return_month rr using (month)
left join refunds_by_sale_month rs using (month)
order by s.month
""").show()

**What to notice:** Both returns happened in February, so dated by return all 105.00 of refunds reduce February. Dated by sale, the
60.00 refund belongs to January, because order 1003 was sold then. Totals over both months agree (1,017.50 −
105.00 = 912.50); the months differ. Finance usually wants refunds by return date (that is when money moved);
merchandising wants them by sale month (how good was January's selling). The two-fact design serves both.

## Case 4: music streaming and royalties

**Prompt.** A streaming service records every play. Questions: plays and listening time per track per day;
royalties per artist, where a track can have several artists sharing the royalty in agreed proportions;
listeners per artist.

**Design.**

- `fact_play`: **one row per play**, with listener, track, start time, seconds played, and the royalty owed
  for the play. Billions of rows: partition by date, cluster by track.
- Tracks and artists are many-to-many, so `bridge_track_artist` holds one row per (track, artist) with a
  **royalty share** that sums to 1 per track (lesson 4).
- Royalties per artist: allocate each play's royalty through the bridge with the share. Listeners per artist:
  count distinct listeners through the bridge **without** weights. A listener of a two-artist track is a listener
  of both artists: full credit is the right answer for this question.
- A daily aggregate by track (plays, seconds, royalties) serves dashboards; distinct listeners cannot be summed
  from it.

**Where it matters**
- Royalties, revenue sharing, multi-touch marketing attribution, commission splits.

**Watch out for**
- Using allocated (weighted) counts for "listeners", or full credit for royalties: each question needs the other method.
- Summing daily distinct listener counts to get monthly listeners.

### Example

Two tracks, one of them a collaboration with a 70/30 split, and five plays.

In [ ]:
con.sql("""
create table bridge_track_artist (track varchar, artist varchar, royalty_share decimal(4, 2));
insert into bridge_track_artist values ('Night Drive', 'Nova', 0.70), ('Night Drive', 'Kite', 0.30), ('Morning', 'Kite', 1.00);

create table fact_play (listener varchar, track varchar, play_date date, seconds int, royalty decimal(8, 4));
insert into fact_play values
    ('u1', 'Night Drive', '2026-03-01', 200, 0.0040), ('u2', 'Night Drive', '2026-03-01', 210, 0.0040),
    ('u1', 'Morning',     '2026-03-01', 180, 0.0040), ('u3', 'Morning',     '2026-03-02', 175, 0.0040),
    ('u1', 'Night Drive', '2026-03-02',  30, 0.0000);   -- skipped after 30 seconds: no royalty
""")
con.sql("""
select coalesce(b.artist, '(all artists)') as artist,
       sum(p.royalty * b.royalty_share)   as royalties_allocated,
       sum(p.royalty)                     as royalties_full_credit,
       count(distinct p.listener)         as listeners
from fact_play p join bridge_track_artist b using (track)
group by rollup (b.artist) order by grouping(b.artist), b.artist
""").show()

**What to notice:** Allocated royalties split each Night Drive play 70/30, and the column adds up to the 0.0160 actually owed (the
total row). Full credit would pay 0.0240, more than exists. For listeners the opposite holds: Kite has three
listeners (u1 and u2 through Night Drive, u3 through Morning) and Nova two, and the total row's 3 is not the sum
of the artists' 5 because u1 and u2 are counted under both. Distinct counts never add up across groups.

## Questions that come up in almost every modeling interview

| Question | A good short answer |
|---|---|
| What is the grain of a table? | What one row represents, stated in one sentence. It is the table's key, and it is decided before any column. |
| Star or snowflake? | Star by default: fewer joins, simpler for users, and columnar compression removes the storage argument. Snowflake a part of a dimension that is large or shared. |
| Why surrogate keys? | History (one row per version), several sources with clashing IDs, unknown members, and insulation from source key changes. |
| How do you keep history of an attribute? | Type 2: new row with a new surrogate key and a half-open validity range; facts join to the version valid at their time. |
| How do you avoid double counting? | Keep fact-to-dimension joins many-to-one, aggregate facts separately before combining them, use bridges with weights for many-to-many, and test uniqueness and totals. |
| What is a factless fact table? | A fact with keys and no measures, recording that something happened or was possible (attendance, promotion coverage). You count its rows. |
| Transaction, periodic snapshot or accumulating snapshot? | Events → transaction; levels over time → periodic snapshot; a process with milestones → accumulating snapshot. |
| What is a conformed dimension? | One dimension shared by several facts with the same keys and values, which makes comparing facts (drill-across) valid. |
| How do you handle late-arriving data? | Partition by event time, reprocess a lookback window idempotently, and resolve dimensions as of the event time; early facts get inferred dimension members. |
| Kimball, Inmon or Data Vault? | Kimball stars to query; Inmon or Data Vault underneath when integration across many sources, audit or change-absorption justify the cost. |
| What would you test? | Grain uniqueness, not-null keys, relationships, accepted values, reconciliation to source, freshness, and SCD invariants. |
| One big table or star? | Model as a star; publish wide tables as marts where measured queries need them. |

For practice with expected answers, continue to the drills.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** An interviewer adds: "now report driver earnings by the city the driver was based in at the time." What changes?

**A.** Driver home city becomes a Type 2 attribute of `dim_driver`, and trips store (or join to) the driver version valid at the trip time. Nothing else changes.

**Q2.** Why is MRR modeled as a monthly snapshot rather than summed from invoices?

**A.** MRR is a level, not a flow. Invoices follow billing cycles (annual plans bill once a year); a snapshot from subscription events gives the recurring amount each customer owes at every month end.

**Q3.** A colleague proposes reducing the sales fact's amount when an item is returned. What do you say?

**A.** The sale and the return are different events with different dates and attributes. Changing the sale rewrites history and loses the return date and reason. Add a return fact linked to the sale line and drill across.

**Q4.** Through the track-artist bridge, should listeners per artist use the royalty share?

**A.** No. Share is for allocating amounts so they add up. A listener of a collaboration listened to every artist on it, so count distinct listeners with full credit, and do not expect the artists' counts to add up.

**Q5.** What is the first thing you write on the whiteboard in a modeling interview, and why?

**A.** The questions the model must answer and the grain sentence of the main fact. Every later decision (dimensions, history, measures, partitioning) follows from them, and they expose misunderstandings early.

## Key takeaways

- Start every design from concrete questions, then process, grain, dimensions, history, measures.
- Keep events you might be asked to count, such as cancelled requests, as fact rows.
- Levels over time (MRR, balances) are periodic snapshots built from events, and are semi-additive.
- New processes such as returns get new facts linked to the original; never rewrite the original fact.
- Bridges serve two questions: weighted allocation for amounts, full credit for reach and distinct counts.
- Prove a design by writing the query for one of the questions against it.

**Next:** practise in the [Data modeling drills](data-modeling-drills.ipynb) and keep the [Data modeling reference](data-modeling-reference.ipynb) open while you work.